# Day 6: Conversation Memory — Remembering What Happened

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> An agent without memory is like a goldfish with a calculator.

Yesterday's agent answered one question and forgot everything. Today we make the
agent **remember** previous questions and answers. This is conversation memory —
the agent's ability to maintain context across multiple turns.

## What You'll Build
- A multi-turn conversation in all 3 frameworks
- The agent remembers: "What was the capital of France?" → "And what's its population?"
- See how each framework handles message history

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 6 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (8 model(s) available)

DAY 6 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Conversation

**Turn 1:** "What is the capital of France?"

**Turn 2:** "And what's the population of that city?" *(refers back to "that city")*

**Turn 3:** "Compare it to the capital of Japan." *(refers back to both capitals)*

If the agent remembers, it answers correctly. If not, it asks "Which city?"

---
## Framework 1: OpenAI Agents SDK

The Runner does **not** maintain session context automatically. Each `Runner.run()`
call is independent — you must build and pass the conversation history yourself.

The pattern: append the user message to a list, pass the full list to `Runner.run()`,
then call `result.to_input_list()` to get back the complete history (including the
agent's response) for the next turn.

In [7]:
from agents import Agent, Runner

model = get_openai_agents_model()

agent = Agent(
    name="Conversationalist",
    instructions="You are a helpful geography assistant. Remember previous questions in this conversation.",
    model=model,
)

# ── Multi-turn conversation ──────────────────────────────
# Runner.run needs the full conversation history each time.
# result.to_input_list() returns all messages including the agent's response.
turns = [
    "What is the capital of France?",
    "And what's the population of that city?",
    "Compare it to the capital of Japan.",
]

print("=" * 60)
print("OPENAI AGENTS SDK — MULTI-TURN")
print("=" * 60)

conversation = []
for i, turn in enumerate(turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    conversation.append({"role": "user", "content": turn})
    result = await Runner.run(agent, conversation)
    print(f"Agent: {result.final_output}")
    conversation = result.to_input_list()

OPENAI AGENTS SDK — MULTI-TURN

--- Turn 1 ---
User: What is the capital of France?


/var/folders/s0/2pw05fm942n5t1b5j7zzd_5m0000gn/T/ipykernel_81001/2115967629.py:29: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  result = await Runner.run(agent, conversation)


Agent: The capital of France is Paris.

--- Turn 2 ---
User: And what's the population of that city?
Agent: As of the latest available data, the population of Paris is approximately 2.2 million people within its administrative limits. However, when considering the whole greater Paris area (which includes towns in the plain of France surrounding Paris), the metropolitan area has a population close to 10 million people.

--- Turn 3 ---
User: Compare it to the capital of Japan.
Agent: The capital of Japan is Tokyo. As of the latest available data, the population of Tokyo is much larger than that of Paris:

- The city proper (in administrative limits) has a population of approximately 9 million people.
- The greater metropolitan area, which includes suburban areas, has a population around 40 million.

So, while Paris has about 2.2 million inhabitants within its city limits, Tokyo's metropolitan area is significantly larger with over 40 million residents.


---
## Framework 2: LangGraph

LangGraph accumulates messages in the state. Each `invoke()` adds to the same
message list. With `MemorySaver`, the state persists across invocations.

In [5]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

model = get_langgraph_model()

# MemorySaver persists state across invocations
memory = MemorySaver()

agent = create_react_agent(
    model,
    tools=[],
    prompt="You are a helpful geography assistant. Remember previous questions.",
    checkpointer=memory,  # THIS enables memory
)

turns = [
    "What is the capital of France?",
    "And what's the population of that city?",
    "Compare it to the capital of Japan.",
]

print("=" * 60)
print("LANGGRAPH — MULTI-TURN (with MemorySaver)")
print("=" * 60)

config = {"configurable": {"thread_id": "geo-conversation-1"}}
for i, turn in enumerate(turns, 1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    result = agent.invoke(
        {"messages": [HumanMessage(content=turn)]},
        config=config,
    )
    last_msg = result["messages"][-1].content
    print(f"Agent: {last_msg}")
    print(f"(State now has {len(result['messages'])} messages)")

/var/folders/s0/2pw05fm942n5t1b5j7zzd_5m0000gn/T/ipykernel_81001/4256999671.py:10: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


LANGGRAPH — MULTI-TURN (with MemorySaver)

--- Turn 1 ---
User: What is the capital of France?
Agent: The capital of France is Paris.
(State now has 2 messages)

--- Turn 2 ---
User: And what's the population of that city?
Agent: As of the latest available data, the population of Paris is approximately 2.2 million people within its administrative limits. However, when considering the entire urban area (which includes satellite towns and outlying suburbs), the estimated population exceeds 10 million.
(State now has 4 messages)

--- Turn 3 ---
User: Compare it to the capital of Japan.
Agent: The capital of Japan is Tokyo. As of the latest data available, the population of Tokyo is significantly larger than that of Paris:

- Population of Tokyo: Approximately 9.28 million people within its administrative limits.
- Population of Paris: Approximately 2.2 million people within its administrative limits.

When considering the entire urban area (which includes satellite towns and outlying subu

---
## Framework 3: CrewAI

CrewAI's `memory=True` on the Agent is designed for semantic search (ChromaDB +
embeddings), **not** for conversation history. Two problems:

1. It requires an embedder (default: OpenAI embeddings). If you're running a local
   model with Ollama and no OpenAI API key, every memory operation fails silently.
2. Memory is **drained when the Crew finishes**. Creating a new Crew per turn means
   zero persistence between turns.

The fair approach for local models: pass conversation context manually in the task
description — the same concept as OpenAI SDK's message list, just as a string.

In [10]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

agent = Agent(
    role="Conversationalist",
    goal="Answer geography questions, using conversation context from previous turns",
    backstory="You are a geography professor who remembers every question in the conversation.",
    llm=llm,
    verbose=True,
)

turns = [
    "What is the capital of France?",
    "And what's the population of that city?",       # refers to Paris from turn 1
    "Compare it to the capital of Japan.",           # refers to Paris + Tokyo
]

print("=" * 60)
print("CREWAI — MULTI-TURN (manual context passing)")
print("=" * 60)

# CrewAI drains memory on Crew finish and memory=True needs an embedder (ChromaDB
# + OpenAI API key). The fair approach: pass conversation history manually in
# the task description, same as OpenAI SDK builds a message list.
conversation_context = ""

for i, turn in enumerate(turns, 1):
    if conversation_context:
        full_prompt = (
            f"Previous conversation:\n{conversation_context}\n\n"
            f"Answer this new question. Use 'that city' or 'it' to refer to "
            f"the subject of the previous answers.\n\nQuestion: {turn}"
        )
    else:
        full_prompt = turn

    task = Task(
        description=full_prompt,
        expected_output="A helpful answer that uses conversation context.",
        agent=agent,
    )
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=True)
    result = await crew.kickoff_async()

    print(f"\n--- Turn {i} ---")
    print(f"User: {turn}")
    print(f"Agent: {result}")

    conversation_context += f"User: {turn}\nAgent: {result}\n"

CREWAI — MULTI-TURN (manual context passing)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b10ac46d-e242-4355-a297-8d669b85447a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the capital of France?                                                                           │
│  ID: ea01a07a-2d52-403b-a732-6f8362763c83                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Conversationalist                                                                                       │
│                                                                                                                 │
│  Task: What is the capital of France?                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Conversationalist                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Sure! To compare Paris (that city) to Tokyo (the capital city of Japan), we can note some differences in       │
│  population and size. Paris's urban area has an approximate population of around 2.3 million people, which is   │
│  significantly larger than the population of the greater Tokyo area, which is closer to 9 million.              │
│  Additionally, while both cities are major capitals with rich cultural histories, their overall layouts and     │
│  geographical features differ greatly due to their distinct climates, topographies, and national identities.    │
│  It's interesting to compare them side by side as you explore these fascinating global metropolises!            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Previous conversation:                                                                                   │
│  User: What is the capital of France?                                                                           │
│  Agent: The capital of France is Paris! If you have any more geography questions or if you need information     │
│  about French places like famous landmarks or regions, feel free to ask. What else would you like to know?      │
│  User: And what's the population of that city?                                                                  │
│  Agent: The capital city of France, Paris, has a population of about 2.3 million in its urban area, so it's     │
│  important to note that numbers can vary slightly depending on classification methods. If you have any other    │
│  questions about Paris or French cities, feel free to ask!                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  Answer this new question. Use 'that city' or 'it' to refer to the subject of the previous answers.             │
│                                                                                                                 │
│  Question: Compare it to the capital of Japan.                                                                  │
│  Agent: Conversationalist                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 51378c5d-191d-4568-8a68-bb151a070f60                                                                       │
│  Final Output: Sure! To compare Paris (that city) to Tokyo (the capital city of Japan), we can note some        │
│  differences in population and size. Paris's urban area has an approximate population of around 2.3 million     │
│  people, which is significantly larger than the population of the greater Tokyo area, which is closer to 9      │
│  million. Additionally, while both cities are major capitals with rich cultural histories, their overall        │
│  layouts and geographical features differ greatly due to their distinct climates, topographies, and national    │
│  identities. It's interesting to compare them side by side as you explore these fascinating global              │
│  metropolises!                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- Turn 3 ---
User: Compare it to the capital of Japan.
Agent: Sure! To compare Paris (that city) to Tokyo (the capital city of Japan), we can note some differences in population and size. Paris's urban area has an approximate population of around 2.3 million people, which is significantly larger than the population of the greater Tokyo area, which is closer to 9 million. Additionally, while both cities are major capitals with rich cultural histories, their overall layouts and geographical features differ greatly due to their distinct climates, topographies, and national identities. It's interesting to compare them side by side as you explore these fascinating global metropolises!


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Comparison: Memory Handling

| Aspect | OpenAI SDK | LangGraph | CrewAI |
|---|---|---|---|
| **How memory works** | Manual: build message list, pass each call | Automatic: checkpointer accumulates state | Manual: pass prior Q&A in task description |
| **API** | `Runner.run(agent, messages)` + `result.to_input_list()` | `checkpointer=MemorySaver()` + `thread_id` | Include conversation as text in `Task(description=)` |
| **State you manage** | A Python list of message dicts | Nothing — the checkpointer handles it | A string of prior Q&A |
| **Works with local models?** | Yes | Yes | Yes (but `memory=True` does NOT — it needs OpenAI embeddings) |
| **Persistent across restarts?** | No (in-memory) | Yes — swap MemorySaver for any DB backend | No (in-memory) |
| **Verdict** | Most code, most control | Least code, most powerful | Most friction for conversations |

**Key insights from debugging:**

1. **OpenAI SDK** does not auto-remember anything. If you call `Runner.run(agent, "single string")`
   every turn, the agent sees only that one message. You must maintain a message list and
   pass it every call. `result.to_input_list()` gives you the updated list including the
   agent's latest response.

2. **LangGraph** is the only framework where memory is truly automatic. Set `checkpointer=MemorySaver()`
   once, give each conversation a `thread_id`, and every `invoke()` call adds to the same state.
   The message count grows visibly (2 → 4 → 6 messages). Swap `MemorySaver` for a database
   checkpointer and you get persistent cross-restart memory.

3. **CrewAI's `memory=True` is misleading** for this use case. It activates semantic search
   (ChromaDB + embeddings), not conversation history. It also requires an OpenAI API key
   for the embedder — it silently fails with local models. And even when it works, memory
   is drained when a Crew finishes, so one-Crew-per-turn gives zero persistence. The
   workaround — passing context manually in the task description — works but is the most
   manual of the three approaches.

## Key Takeaway

Memory is what turns a one-shot Q&A into a conversation. All three frameworks can do it,
but the developer experience differs dramatically:

- **LangGraph** is the clear winner. One line (`checkpointer=MemorySaver()`), a thread ID,
  and the framework accumulates messages automatically. You can see the state growing
  (2 → 4 → 6 messages). Swap the backend for a database and you get persistence across
  restarts. This is production-grade memory with zero boilerplate.

- **OpenAI SDK** works fine but is entirely manual. You maintain a list of message dicts,
  pass it to every `Runner.run()` call, and use `result.to_input_list()` to capture
  responses. More code than LangGraph, but you have full control over exactly what's in
  the context window.

- **CrewAI** has the most friction. The `memory=True` flag sounds like it solves this, but
  it's designed for semantic search (ChromaDB + embeddings), not conversation history.
  It requires an OpenAI API key for the embedder, so it silently fails with local models
  like Ollama. And even when it works, memory is drained when a Crew finishes. The practical
  workaround is manual context passing in task descriptions — functional but verbose.

**Bottom line:** If conversation memory matters to your application, LangGraph is the
only framework that handles it properly out of the box. OpenAI SDK and CrewAI both
require you to build your own scaffolding.

---
